### With Feed Forward Neural Network

In [ ]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE

import random
torch.manual_seed(7)
np.random.seed(7)
random.seed(7)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

file_path = 'Java1.csv'
data = pd.read_csv(file_path)

data['Body'] = data['Body'].fillna('').astype(str)
data['Tags'] = data['Tags'].fillna('').astype(str)
data['Title'] = data['Title'].fillna('').astype(str)

data['concatenated_text'] = data['Body'] + " " + data['Tags'] + " " + data['Title']


data['concatenated_text'] = data['Body'] + " " + data['Tags'] + " " + data['Title']

label_mapping = {'Basic': 0, 'Intermediate': 1, 'Advanced': 2}
data['encoded_labels'] = data['Label'].map(label_mapping)
labels = data['encoded_labels'].values

data['num_tags'] = data['Tags'].apply(lambda x: len(x.split(',')))  

numerical_cols.append('num_tags')

print(data[['Tags', 'num_tags']].head())

numerical_cols = [
    'View Count', 'Answer Count',
    'Score', 'Interval from first', 'Interval from accepted',
    'Total count urls and imgs', 'LOC', 'Question_Length', 'num_tags'
]


combined_texts = data['concatenated_text'].tolist()
numerical_data = data[numerical_cols].values
labels = data['Label'].map(label_mapping).values 

X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    combined_texts, numerical_data, labels,
    test_size=0.20, random_state=42, stratify=labels
)

imputer = SimpleImputer(strategy='mean')
X_num_train = imputer.fit_transform(X_num_train)
X_num_test = imputer.transform(X_num_test)

scaler = StandardScaler()
X_num_train = scaler.fit_transform(X_num_train)
X_num_test = scaler.transform(X_num_test)


model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModel.from_pretrained(model_name)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)
model = get_peft_model(base_model, lora_config)
model.to(device)

class StackOverflowDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        inputs = tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            return_tensors="pt",
            max_length=512
        )
        if "token_type_ids" in inputs:
            del inputs["token_type_ids"]
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = StackOverflowDataset(X_text_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)


class SBERTWithClassification(nn.Module):
    def __init__(self, base_model, num_classes):
        super(SBERTWithClassification, self).__init__()
        self.base_model = base_model
        self.classifier = nn.Linear(384, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_embedding)
        return logits

num_classes = len(label_mapping)
classifier_model = SBERTWithClassification(model, num_classes).to(device)


loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(classifier_model.parameters(), lr=5e-5)

num_epochs = 3
for epoch in range(num_epochs):
    classifier_model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        outputs = classifier_model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss / len(train_loader):.4f}")


def extract_embeddings(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=512).to(device)
    if "token_type_ids" in inputs:
        del inputs["token_type_ids"]
    with torch.no_grad():
        outputs = classifier_model.base_model(**inputs)
    return outputs.last_hidden_state[:, 0, :].cpu().numpy()

train_embeddings = np.array([extract_embeddings([text])[0] for text in X_text_train])
test_embeddings = np.array([extract_embeddings([text])[0] for text in X_text_test])

X_train_combined = np.hstack((train_embeddings, X_num_train))
X_test_combined = np.hstack((test_embeddings, X_num_test))

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_combined, y_train)

class FeedForwardNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, output_dim=3):
        super(FeedForwardNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, output_dim)
        )

    def forward(self, x):
        return self.net(x)

X_train_tensor = torch.tensor(X_train_resampled, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_resampled, dtype=torch.long).to(device)
X_test_tensor = torch.tensor(X_test_combined, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long).to(device)

input_dim = X_train_resampled.shape[1]
model_fnn = FeedForwardNN(input_dim=input_dim, output_dim=num_classes).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_fnn.parameters(), lr=1e-4)

epochs = 5
batch_size = 64

for epoch in range(epochs):
    model_fnn.train()
    epoch_loss = 0
    permutation = torch.randperm(X_train_tensor.size(0))
    
    for i in range(0, X_train_tensor.size(0), batch_size):
        indices = permutation[i:i + batch_size]
        batch_x, batch_y = X_train_tensor[indices], y_train_tensor[indices]

        optimizer.zero_grad()
        outputs = model_fnn(batch_x)
        loss = loss_fn(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}")

model_fnn.eval()
with torch.no_grad():
    y_pred_logits = model_fnn(X_test_tensor)
    y_pred = torch.argmax(y_pred_logits, dim=1).cpu().numpy()

print(classification_report(y_test, y_pred, target_names=label_mapping.keys(), digits=4))



In [ ]:
import itertools
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, accuracy_score
from torch.utils.data import DataLoader, TensorDataset


param_grid = {
    'hidden_dim': [64, 128, 256],
    'dropout': [0.1,0.2,0.3, 0.5],
    'learning_rate': [1e-3, 1e-4, 1e-5],
    'batch_size': [16, 32, 64],
    'num_hidden_layers': [1, 2, 3, 5],
    'epochs': [5, 10, 20, 50] 
}

param_combinations = list(itertools.product(*param_grid.values()))
param_keys = list(param_grid.keys())

best_accuracy = 0
best_params = {}

for combo in param_combinations:
    params = dict(zip(param_keys, combo))
    print(f"\n🧪 Testing config: {params}")

    class FeedforwardNN(nn.Module):
        def __init__(self, input_dim, hidden_dim, output_dim, dropout, num_hidden_layers):
            super(FeedforwardNN, self).__init__()
            layers = [nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)]
            for _ in range(num_hidden_layers - 1):
                layers += [nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)]
            layers.append(nn.Linear(hidden_dim, output_dim))
            self.model = nn.Sequential(*layers)

        def forward(self, x):
            return self.model(x)

    model = FeedforwardNN(
        input_dim=X_train_resampled.shape[1],
        hidden_dim=params['hidden_dim'],
        output_dim=len(label_mapping),
        dropout=params['dropout'],
        num_hidden_layers=params['num_hidden_layers']
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])

    train_dataset = TensorDataset(torch.tensor(X_train_resampled, dtype=torch.float32),
                                  torch.tensor(y_train_resampled, dtype=torch.long))
    test_dataset = TensorDataset(torch.tensor(X_test_combined, dtype=torch.float32),
                                 torch.tensor(y_test, dtype=torch.long))

    train_loader = DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

    for epoch in range(params['epochs']):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()

    model.eval()
    all_preds = []
    with torch.no_grad():
        for X_batch, _ in test_loader:
            output = model(X_batch)
            preds = torch.argmax(output, dim=1)
            all_preds.extend(preds.cpu().numpy())

    acc = accuracy_score(y_test, all_preds)
    print(f"🔍 Accuracy: {acc:.4f}")
    print(classification_report(y_test, all_preds, target_names=label_mapping.keys(), digits=4))

    if acc > best_accuracy:
        best_accuracy = acc
        best_params = params

print("\n🎯 Best Configuration:")
print(best_params)
print(f"✅ Best Accuracy: {best_accuracy:.4f}")



🧪 Testing config: {'hidden_dim': 64, 'dropout': 0.1, 'learning_rate': 0.001, 'batch_size': 16, 'num_hidden_layers': 1, 'epochs': 5}
🔍 Accuracy: 0.6507
              precision    recall  f1-score   support

       Basic     0.7983    0.7661    0.7819       124
Intermediate     0.5231    0.5000    0.5113        68
    Advanced     0.2800    0.4118    0.3333        17

    accuracy                         0.6507       209
   macro avg     0.5338    0.5593    0.5422       209
weighted avg     0.6666    0.6507    0.6574       209


🧪 Testing config: {'hidden_dim': 64, 'dropout': 0.1, 'learning_rate': 0.001, 'batch_size': 16, 'num_hidden_layers': 1, 'epochs': 10}
🔍 Accuracy: 0.6316
              precision    recall  f1-score   support

       Basic     0.7712    0.7339    0.7521       124
Intermediate     0.4932    0.5294    0.5106        68
    Advanced     0.2778    0.2941    0.2857        17

    accuracy                         0.6316       209
   macro avg     0.5140    0.5191    0.516

/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


🔍 Accuracy: 0.6316
              precision    recall  f1-score   support

       Basic     0.6294    1.0000    0.7726       124
Intermediate     0.7500    0.0441    0.0833        68
    Advanced     0.6250    0.2941    0.4000        17

    accuracy                         0.6316       209
   macro avg     0.6681    0.4461    0.4186       209
weighted avg     0.6683    0.6316    0.5180       209


🧪 Testing config: {'hidden_dim': 64, 'dropout': 0.1, 'learning_rate': 1e-05, 'batch_size': 16, 'num_hidden_layers': 2, 'epochs': 20}
🔍 Accuracy: 0.6364
              precision    recall  f1-score   support

       Basic     0.7244    0.9113    0.8071       124
Intermediate     0.5500    0.1618    0.2500        68
    Advanced     0.2727    0.5294    0.3600        17

    accuracy                         0.6364       209
   macro avg     0.5157    0.5342    0.4724       209
weighted avg     0.6309    0.6364    0.5895       209


🧪 Testing config: {'hidden_dim': 64, 'dropout': 0.1, 'learning_ra

/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


🔍 Accuracy: 0.6124
              precision    recall  f1-score   support

       Basic     0.7800    0.6290    0.6964       124
Intermediate     0.4587    0.7353    0.5650        68
    Advanced     0.0000    0.0000    0.0000        17

    accuracy                         0.6124       209
   macro avg     0.4129    0.4548    0.4205       209
weighted avg     0.6120    0.6124    0.5970       209


🧪 Testing config: {'hidden_dim': 64, 'dropout': 0.1, 'learning_rate': 1e-05, 'batch_size': 16, 'num_hidden_layers': 3, 'epochs': 20}


...

/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


🔍 Accuracy: 0.3254
              precision    recall  f1-score   support

       Basic     0.0000    0.0000    0.0000       124
Intermediate     0.3254    1.0000    0.4910        68
    Advanced     0.0000    0.0000    0.0000        17

    accuracy                         0.3254       209
   macro avg     0.1085    0.3333    0.1637       209
weighted avg     0.1059    0.3254    0.1597       209


🧪 Testing config: {'hidden_dim': 256, 'dropout': 0.5, 'learning_rate': 1e-05, 'batch_size': 64, 'num_hidden_layers': 5, 'epochs': 50}


/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


🔍 Accuracy: 0.6842
              precision    recall  f1-score   support

       Basic     0.7556    0.8226    0.7876       124
Intermediate     0.5541    0.6029    0.5775        68
    Advanced     0.0000    0.0000    0.0000        17

    accuracy                         0.6842       209
   macro avg     0.4365    0.4752    0.4550       209
weighted avg     0.6285    0.6842    0.6552       209


🎯 Best Configuration:
{'hidden_dim': 64, 'dropout': 0.2, 'learning_rate': 1e-05, 'batch_size': 32, 'num_hidden_layers': 2, 'epochs': 20}
✅ Best Accuracy: 0.7129


/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
import itertools
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, accuracy_score
from torch.utils.data import DataLoader, TensorDataset

param_grid = {
    'hidden_dim': [64, 128, 256],
    'dropout': [0.1,0.2,0.3, 0.5],
    'learning_rate': [1e-3, 1e-4, 1e-5],
    'batch_size': [16, 32, 64],
    'num_hidden_layers': [1, 2, 3, 5],
    'epochs': [5, 10, 20, 50]
}

param_combinations = list(itertools.product(*param_grid.values()))
param_keys = list(param_grid.keys())

best_accuracy = 0
best_params = {}

for combo in param_combinations:
    params = dict(zip(param_keys, combo))
    print(f"\n🧪 Testing config: {params}")

    class FeedforwardNN(nn.Module):
        def __init__(self, input_dim, hidden_dim, output_dim, dropout, num_hidden_layers):
            super(FeedforwardNN, self).__init__()
            layers = [nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)]
            for _ in range(num_hidden_layers - 1):
                layers += [nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)]
            layers.append(nn.Linear(hidden_dim, output_dim))
            self.model = nn.Sequential(*layers)

        def forward(self, x):
            return self.model(x)

    model = FeedforwardNN(
        input_dim=X_train_resampled.shape[1],
        hidden_dim=params['hidden_dim'],
        output_dim=len(label_mapping),
        dropout=params['dropout'],
        num_hidden_layers=params['num_hidden_layers']
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])

    train_dataset = TensorDataset(torch.tensor(X_train_resampled, dtype=torch.float32),
                                  torch.tensor(y_train_resampled, dtype=torch.long))
    test_dataset = TensorDataset(torch.tensor(X_test_combined, dtype=torch.float32),
                                 torch.tensor(y_test, dtype=torch.long))

    train_loader = DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

    for epoch in range(params['epochs']):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()

    model.eval()
    all_preds = []
    with torch.no_grad():
        for X_batch, _ in test_loader:
            output = model(X_batch)
            preds = torch.argmax(output, dim=1)
            all_preds.extend(preds.cpu().numpy())

    acc = accuracy_score(y_test, all_preds)
    print(f"🔍 Accuracy: {acc:.4f}")
    print(classification_report(y_test, all_preds, target_names=label_mapping.keys(), digits=4))

    if acc > best_accuracy:
        best_accuracy = acc
        best_params = params

print("\n🎯 Best Configuration:")
print(best_params)
print(f"✅ Best Accuracy: {best_accuracy:.4f}")



🧪 Testing config: {'hidden_dim': 64, 'dropout': 0.1, 'learning_rate': 0.001, 'batch_size': 16, 'num_hidden_layers': 1, 'epochs': 5}
🔍 Accuracy: 0.6411
              precision    recall  f1-score   support

       Basic     0.8526    0.6532    0.7397       124
Intermediate     0.5052    0.7206    0.5939        68
    Advanced     0.2353    0.2353    0.2353        17

    accuracy                         0.6411       209
   macro avg     0.5310    0.5364    0.5230       209
weighted avg     0.6894    0.6411    0.6513       209


🧪 Testing config: {'hidden_dim': 64, 'dropout': 0.1, 'learning_rate': 0.001, 'batch_size': 16, 'num_hidden_layers': 1, 'epochs': 10}
🔍 Accuracy: 0.6364
              precision    recall  f1-score   support

       Basic     0.8056    0.7016    0.7500       124
Intermediate     0.4941    0.6176    0.5490        68
    Advanced     0.2500    0.2353    0.2424        17

    accuracy                         0.6364       209
   macro avg     0.5166    0.5182    0.513

/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


🔍 Accuracy: 0.6077
              precision    recall  f1-score   support

       Basic     0.7692    0.8065    0.7874       124
Intermediate     0.4054    0.2206    0.2857        68
    Advanced     0.2857    0.7059    0.4068        17

    accuracy                         0.6077       209
   macro avg     0.4868    0.5776    0.4933       209
weighted avg     0.6115    0.6077    0.5932       209


🧪 Testing config: {'hidden_dim': 64, 'dropout': 0.1, 'learning_rate': 0.0001, 'batch_size': 64, 'num_hidden_layers': 5, 'epochs': 20}
🔍 Accuracy: 0.6699
              precision    recall  f1-score   support

       Basic     0.8034    0.7581    0.7801       124
Intermediate     0.5143    0.5294    0.5217        68
    Advanced     0.4545    0.5882    0.5128        17

    accuracy                         0.6699       209
   macro avg     0.5907    0.6252    0.6049       209
weighted avg     0.6810    0.6699    0.6743       209


🧪 Testing config: {'hidden_dim': 64, 'dropout': 0.1, 'learning_r

/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


🔍 Accuracy: 0.4641
              precision    recall  f1-score   support

       Basic     0.8429    0.4758    0.6082       124
Intermediate     0.3021    0.4265    0.3537        68
    Advanced     0.2093    0.5294    0.3000        17

    accuracy                         0.4641       209
   macro avg     0.4514    0.4772    0.4206       209
weighted avg     0.6154    0.4641    0.5003       209


🧪 Testing config: {'hidden_dim': 64, 'dropout': 0.1, 'learning_rate': 1e-05, 'batch_size': 16, 'num_hidden_layers': 2, 'epochs': 20}
🔍 Accuracy: 0.6459
              precision    recall  f1-score   support

       Basic     0.6667    0.9839    0.7948       124
Intermediate     0.8333    0.0735    0.1351        68
    Advanced     0.4000    0.4706    0.4324        17

    accuracy                         0.6459       209
   macro avg     0.6333    0.5093    0.4541       209
weighted avg     0.6992    0.6459    0.5507       209


🧪 Testing config: {'hidden_dim': 64, 'dropout': 0.1, 'learning_ra

...

🔍 Accuracy: 0.3254
              precision    recall  f1-score   support

       Basic     0.0000    0.0000    0.0000       124
Intermediate     0.3254    1.0000    0.4910        68
    Advanced     0.0000    0.0000    0.0000        17

    accuracy                         0.3254       209
   macro avg     0.1085    0.3333    0.1637       209
weighted avg     0.1059    0.3254    0.1597       209


🧪 Testing config: {'hidden_dim': 256, 'dropout': 0.5, 'learning_rate': 1e-05, 'batch_size': 64, 'num_hidden_layers': 5, 'epochs': 20}


/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


🔍 Accuracy: 0.5933
              precision    recall  f1-score   support

       Basic     0.5933    1.0000    0.7447       124
Intermediate     0.0000    0.0000    0.0000        68
    Advanced     0.0000    0.0000    0.0000        17

    accuracy                         0.5933       209
   macro avg     0.1978    0.3333    0.2482       209
weighted avg     0.3520    0.5933    0.4419       209


🧪 Testing config: {'hidden_dim': 256, 'dropout': 0.5, 'learning_rate': 1e-05, 'batch_size': 64, 'num_hidden_layers': 5, 'epochs': 50}


/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


🔍 Accuracy: 0.4258
              precision    recall  f1-score   support

       Basic     0.7732    0.6048    0.6787       124
Intermediate     1.0000    0.0147    0.0290        68
    Advanced     0.1171    0.7647    0.2031        17

    accuracy                         0.4258       209
   macro avg     0.6301    0.4614    0.3036       209
weighted avg     0.7936    0.4258    0.4286       209


🎯 Best Configuration:
{'hidden_dim': 256, 'dropout': 0.3, 'learning_rate': 0.001, 'batch_size': 64, 'num_hidden_layers': 5, 'epochs': 5}
✅ Best Accuracy: 0.7368


🎯 Best Configuration:
{'hidden_dim': 256, 'dropout': 0.3, 'learning_rate': 0.001, 'batch_size': 64, 'num_hidden_layers': 5, 'epochs': 5}
✅ Best Accuracy: 0.7368